# Feature Engineering and Data Preparation

This notebook covers:
1. **Loading cleaned data** from the data exploration notebook
2. **Feature selection** based on EDA findings
3. **Scaling comparison** (StandardScaler vs RobustScaler)
4. **Train/Test split** with stratification
5. **Resampling methods** (RUS, ROS, SMOTE)
6. **Final data preparation** for modeling

---

## Data Passing Between Notebooks

**Method 1: CSV Files (Recommended for DataFrames)**
- Save cleaned data as CSV in notebook 01
- Load CSV in notebook 02
- Pros: Human-readable, version control friendly
- Cons: Larger file size, slower for very large datasets

**Method 2: Pickle Files (Recommended for Complex Objects)**
- Save preprocessed data as pickle in notebook 01
- Load pickle in notebook 02
- Pros: Fast, preserves data types exactly
- Cons: Not human-readable, Python-specific

**Method 3: Re-run Cells (For Development)**
- Re-run data loading and preprocessing cells
- Pros: No intermediate files
- Cons: Slower, requires all dependencies

**For this project:** We'll use CSV for the cleaned dataset and pickle for preprocessed/scaled datasets.


## 1. Import Libraries and Load Data


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


In [ ]:
# Load cleaned dataset from notebook 01
# Option 1: Load from saved CSV (recommended)
data_path = '../data/processed/creditcard_cleaned.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✓ Loaded cleaned data from CSV")
    print(f"  Shape: {df.shape}")
    print(f"  Class distribution:")
    print(df['Class'].value_counts())
else:
    # Option 2: Load original and apply cleaning (if CSV not available)
    print("⚠ Processed data not found. Loading original data...")
    df = pd.read_csv('../data/creditcard.csv')
    # Apply duplicate removal (as done in notebook 01)
    df = df.drop_duplicates().copy()
    print(f"  Shape after cleaning: {df.shape}")
    print(f"  Class distribution:")
    print(df['Class'].value_counts())


## 2. Feature Selection

Based on EDA findings from notebook 01, we'll select features for modeling.


In [ ]:
# Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())
print(f"\nFeatures: {list(X.columns)}")

# Note: From EDA, we know:
# - V1-V28 are PCA components (already standardized)
# - Time and Amount may need scaling
# - Top correlated features: V17, V14, V12, V11, V4, V10, V16, V3, V7


## 3. Train/Test Split

We'll create a stratified train/test split to maintain class distribution.


In [ ]:
# Stratified train/test split (80/20)
# This ensures both sets maintain the same class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Maintains class distribution
)

print("="*70)
print("TRAIN/TEST SPLIT SUMMARY")
print("="*70)
print(f"\nTraining set:")
print(f"  Shape: {X_train.shape}")
print(f"  Class distribution:")
print(y_train.value_counts())
print(f"  Fraud rate: {y_train.mean() * 100:.2f}%")

print(f"\nTest set:")
print(f"  Shape: {X_test.shape}")
print(f"  Class distribution:")
print(y_test.value_counts())
print(f"  Fraud rate: {y_test.mean() * 100:.2f}%")

print(f"\n✓ Split completed successfully!")
print(f"  Training samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")


## 4. Scaling Comparison: StandardScaler vs RobustScaler

We'll compare both scalers to determine which performs better for our fraud detection task.


In [ ]:
# Initialize scalers
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

# Fit and transform training data
X_train_standard = standard_scaler.fit_transform(X_train)
X_train_robust = robust_scaler.fit_transform(X_train)

# Transform test data (using training fit)
X_test_standard = standard_scaler.transform(X_test)
X_test_robust = robust_scaler.transform(X_test)

# Convert back to DataFrames for easier handling
X_train_standard = pd.DataFrame(X_train_standard, columns=X_train.columns, index=X_train.index)
X_train_robust = pd.DataFrame(X_train_robust, columns=X_train.columns, index=X_train.index)
X_test_standard = pd.DataFrame(X_test_standard, columns=X_test.columns, index=X_test.index)
X_test_robust = pd.DataFrame(X_test_robust, columns=X_test.columns, index=X_test.index)

print("✓ Scaling completed for both methods")
print(f"  StandardScaler: Mean={X_train_standard.mean().mean():.6f}, Std={X_train_standard.std().mean():.6f}")
print(f"  RobustScaler: Median={X_train_robust.median().median():.6f}, IQR-based scaling")


### 4.1 Compare Scalers Using Logistic Regression

We'll use Logistic Regression as a baseline to compare scaler performance.


In [ ]:
# Train Logistic Regression with StandardScaler
lr_standard = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
lr_standard.fit(X_train_standard, y_train)
y_pred_proba_standard = lr_standard.predict_proba(X_test_standard)[:, 1]
roc_auc_standard = roc_auc_score(y_test, y_pred_proba_standard)

# Train Logistic Regression with RobustScaler
lr_robust = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
lr_robust.fit(X_train_robust, y_train)
y_pred_proba_robust = lr_robust.predict_proba(X_test_robust)[:, 1]
roc_auc_robust = roc_auc_score(y_test, y_pred_proba_robust)

# Compare results
print("="*70)
print("SCALER COMPARISON RESULTS")
print("="*70)
print(f"\nStandardScaler:")
print(f"  ROC-AUC Score: {roc_auc_standard:.4f}")

print(f"\nRobustScaler:")
print(f"  ROC-AUC Score: {roc_auc_robust:.4f}")

print(f"\n{'='*70}")
if roc_auc_robust > roc_auc_standard:
    print(f"✓ RobustScaler performs better (difference: {roc_auc_robust - roc_auc_standard:.4f})")
    print(f"  Recommendation: Use RobustScaler for all models")
    best_scaler = robust_scaler
    best_scaler_name = "RobustScaler"
else:
    print(f"✓ StandardScaler performs better (difference: {roc_auc_standard - roc_auc_robust:.4f})")
    print(f"  Recommendation: Use StandardScaler for all models")
    best_scaler = standard_scaler
    best_scaler_name = "StandardScaler"
print("="*70)


### 4.2 Visualize Scaler Comparison


In [ ]:
# Visualize distribution differences for a key feature (Amount)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Original Amount distribution
axes[0, 0].hist(X_train['Amount'], bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0, 0].set_title('Original Amount Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Amount')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# StandardScaler transformed
axes[0, 1].hist(X_train_standard['Amount'], bins=50, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_title('StandardScaler: Amount Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Amount (Standardized)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# RobustScaler transformed
axes[1, 0].hist(X_train_robust['Amount'], bins=50, alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_title('RobustScaler: Amount Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Amount (Robust Scaled)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

# ROC-AUC comparison
axes[1, 1].bar(['StandardScaler', 'RobustScaler'], 
               [roc_auc_standard, roc_auc_robust],
               color=['green', 'orange'], alpha=0.7, edgecolor='black')
axes[1, 1].set_title('ROC-AUC Comparison (Logistic Regression)', fontweight='bold')
axes[1, 1].set_ylabel('ROC-AUC Score')
axes[1, 1].set_ylim([min(roc_auc_standard, roc_auc_robust) - 0.01, 
                     max(roc_auc_standard, roc_auc_robust) + 0.01])
axes[1, 1].grid(True, alpha=0.3, axis='y')
# Add value labels
for i, (name, score) in enumerate(zip(['StandardScaler', 'RobustScaler'], 
                                       [roc_auc_standard, roc_auc_robust])):
    axes[1, 1].text(i, score, f'{score:.4f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Scaler Comparison Analysis', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


## 5. Apply Best Scaler

We'll use the best performing scaler for all subsequent steps.


In [ ]:
# Use the best scaler (determined above)
if best_scaler_name == "RobustScaler":
    X_train_scaled = X_train_robust.copy()
    X_test_scaled = X_test_robust.copy()
    scaler = robust_scaler
else:
    X_train_scaled = X_train_standard.copy()
    X_test_scaled = X_test_standard.copy()
    scaler = standard_scaler

print(f"✓ Using {best_scaler_name} for all models")
print(f"  Training set shape: {X_train_scaled.shape}")
print(f"  Test set shape: {X_test_scaled.shape}")


## 6. Resampling Methods

We'll apply different resampling techniques to handle class imbalance. We'll keep the original data for class_weight models and create resampled versions for comparison.


In [ ]:
# Store original distribution (for class_weight models)
print("="*70)
print("ORIGINAL TRAINING SET (for class_weight models)")
print("="*70)
print(f"Shape: {X_train_scaled.shape}")
print(f"Class distribution: {Counter(y_train)}")
print(f"Imbalance ratio: {Counter(y_train)[0] / Counter(y_train)[1]:.1f}:1")


### 6.1 Random Under Sampling (RUS)


In [ ]:
# Apply Random Under Sampling
rus = RandomUnderSampler(random_state=42, sampling_strategy='auto')
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled.copy(), y_train.copy())

print("="*70)
print("RANDOM UNDER SAMPLING (RUS)")
print("="*70)
print(f"Before RUS: {X_train_scaled.shape[0]:,} samples")
print(f"After RUS: {X_train_rus.shape[0]:,} samples")
print(f"Samples removed: {X_train_scaled.shape[0] - X_train_rus.shape[0]:,}")
print(f"Data loss: {((X_train_scaled.shape[0] - X_train_rus.shape[0]) / X_train_scaled.shape[0]) * 100:.2f}%")
print(f"\nClass distribution: {Counter(y_train_rus)}")
print(f"Imbalance ratio: {Counter(y_train_rus)[0] / Counter(y_train_rus)[1]:.1f}:1")


### 6.2 Random Over Sampling (ROS)


In [ ]:
# Apply Random Over Sampling
ros = RandomOverSampler(random_state=42, sampling_strategy='auto')
X_train_ros, y_train_ros = ros.fit_resample(X_train_scaled.copy(), y_train.copy())

print("="*70)
print("RANDOM OVER SAMPLING (ROS)")
print("="*70)
print(f"Before ROS: {X_train_scaled.shape[0]:,} samples")
print(f"After ROS: {X_train_ros.shape[0]:,} samples")
print(f"Samples added: {X_train_ros.shape[0] - X_train_scaled.shape[0]:,}")
print(f"Data increase: {((X_train_ros.shape[0] - X_train_scaled.shape[0]) / X_train_scaled.shape[0]) * 100:.2f}%")
print(f"\nClass distribution: {Counter(y_train_ros)}")
print(f"Imbalance ratio: {Counter(y_train_ros)[0] / Counter(y_train_ros)[1]:.1f}:1")


### 6.3 SMOTE (Synthetic Minority Oversampling)


In [ ]:
# Apply SMOTE
smote = SMOTE(random_state=42, sampling_strategy='auto')
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled.copy(), y_train.copy())

print("="*70)
print("SMOTE (Synthetic Minority Oversampling)")
print("="*70)
print(f"Before SMOTE: {X_train_scaled.shape[0]:,} samples")
print(f"After SMOTE: {X_train_smote.shape[0]:,} samples")
print(f"Samples added: {X_train_smote.shape[0] - X_train_scaled.shape[0]:,}")
print(f"Data increase: {((X_train_smote.shape[0] - X_train_scaled.shape[0]) / X_train_scaled.shape[0]) * 100:.2f}%")
print(f"\nClass distribution: {Counter(y_train_smote)}")
print(f"Imbalance ratio: {Counter(y_train_smote)[0] / Counter(y_train_smote)[1]:.1f}:1")


### 6.4 Resampling Comparison Visualization


In [ ]:
# Create comparison DataFrame
comparison_data = {
    'Method': ['Original', 'RUS', 'ROS', 'SMOTE'],
    'Class_0_Count': [
        Counter(y_train)[0],
        Counter(y_train_rus)[0],
        Counter(y_train_ros)[0],
        Counter(y_train_smote)[0]
    ],
    'Class_1_Count': [
        Counter(y_train)[1],
        Counter(y_train_rus)[1],
        Counter(y_train_ros)[1],
        Counter(y_train_smote)[1]
    ],
    'Total_Samples': [
        len(y_train),
        len(y_train_rus),
        len(y_train_ros),
        len(y_train_smote)
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Imbalance_Ratio'] = comparison_df['Class_0_Count'] / comparison_df['Class_1_Count']

print("="*70)
print("RESAMPLING METHODS COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))


In [ ]:
# Visualize resampling comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Count comparison
ax1 = axes[0, 0]
x_pos = np.arange(len(comparison_df))
width = 0.35
ax1.bar(x_pos - width/2, comparison_df['Class_0_Count'], width,
        label='Class 0 (Normal)', color='blue', alpha=0.7)
ax1.bar(x_pos + width/2, comparison_df['Class_1_Count'], width,
        label='Class 1 (Fraud)', color='red', alpha=0.7)
ax1.set_xlabel('Method')
ax1.set_ylabel('Count')
ax1.set_title('Class Distribution: Count Comparison', fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(comparison_df['Method'], rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_yscale('log')

# 2. Total samples comparison
ax2 = axes[0, 1]
bars = ax2.bar(comparison_df['Method'], comparison_df['Total_Samples'],
               color=['gray', 'orange', 'green', 'purple'], alpha=0.7)
ax2.set_ylabel('Total Samples')
ax2.set_title('Total Dataset Size by Method', fontweight='bold')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, comparison_df['Total_Samples']):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 3. Imbalance ratio comparison
ax3 = axes[1, 0]
bars = ax3.bar(comparison_df['Method'], comparison_df['Imbalance_Ratio'],
               color=['gray', 'orange', 'green', 'purple'], alpha=0.7)
ax3.set_ylabel('Imbalance Ratio (Class 0 : Class 1)')
ax3.set_title('Imbalance Ratio by Method', fontweight='bold')
ax3.axhline(y=1.0, color='green', linestyle='--', linewidth=2,
            label='Perfectly Balanced (1:1)', alpha=0.7)
ax3.grid(True, alpha=0.3, axis='y')
ax3.legend()
for bar, val in zip(bars, comparison_df['Imbalance_Ratio']):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:.1f}:1', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Feature distribution comparison (using V14 as example)
ax4 = axes[1, 1]
key_feature = 'V14'  # One of the most correlated features
ax4.hist(X_train_scaled[y_train == 0][key_feature], bins=50, alpha=0.5,
         label='Original Normal', color='blue', density=True)
ax4.hist(X_train_scaled[y_train == 1][key_feature], bins=50, alpha=0.5,
         label='Original Fraud', color='red', density=True)
ax4.hist(X_train_smote[y_train_smote == 1][key_feature], bins=50, alpha=0.3,
         label='SMOTE Fraud', color='orange', density=True, linestyle='--')
ax4.set_xlabel(key_feature)
ax4.set_ylabel('Density')
ax4.set_title(f'{key_feature} Distribution: Original vs SMOTE', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.suptitle('Resampling Methods Comparison', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
# Create directories for saving processed data
os.makedirs('../data/processed', exist_ok=True)

# Save original scaled data (for class_weight models)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save resampled datasets
X_train_rus.to_csv('../data/processed/X_train_rus.csv', index=False)
y_train_rus_series = pd.Series(y_train_rus)
y_train_rus_series.to_csv('../data/processed/y_train_rus.csv', index=False)

X_train_ros.to_csv('../data/processed/X_train_ros.csv', index=False)
y_train_ros_series = pd.Series(y_train_ros)
y_train_ros_series.to_csv('../data/processed/y_train_ros.csv', index=False)

X_train_smote.to_csv('../data/processed/X_train_smote.csv', index=False)
y_train_smote_series = pd.Series(y_train_smote)
y_train_smote_series.to_csv('../data/processed/y_train_smote.csv', index=False)

# Save scaler for use in modeling notebook
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("="*70)
print("DATA SAVED SUCCESSFULLY")
print("="*70)
print("\nSaved files:")
print("  - X_train_scaled.csv (original, for class_weight models)")
print("  - X_test_scaled.csv")
print("  - y_train.csv")
print("  - y_test.csv")
print("  - X_train_rus.csv (Random Under Sampling)")
print("  - y_train_rus.csv")
print("  - X_train_ros.csv (Random Over Sampling)")
print("  - y_train_ros.csv")
print("  - X_train_smote.csv (SMOTE)")
print("  - y_train_smote.csv")
print("  - scaler.pkl (fitted scaler)")
print("\n✓ All datasets ready for modeling notebook!")


## 8. Summary and Next Steps

### Summary of Feature Engineering:
1. ✓ **Data Loaded**: Cleaned dataset from notebook 01
2. ✓ **Features Selected**: All features (V1-V28, Time, Amount)
3. ✓ **Scaler Chosen**: {best_scaler_name} (based on Logistic Regression performance)
4. ✓ **Train/Test Split**: 80/20 stratified split
5. ✓ **Resampling Methods**: Original, RUS, ROS, SMOTE datasets prepared
6. ✓ **Data Saved**: All prepared datasets saved for modeling notebook

### Next Steps (Notebook 03):
1. Load prepared datasets
2. Train baseline models:
   - Logistic Regression (with class_weight and with resampling methods)
   - Random Forest (with class_weight and with resampling methods)
3. Evaluate and compare models
4. Perform feature importance analysis
5. Hyperparameter tuning on top models
6. Final model selection

### Available Datasets for Modeling:
- **Original (imbalanced)**: Use with `class_weight='balanced'`
- **RUS**: Balanced, reduced dataset
- **ROS**: Balanced, duplicated samples
- **SMOTE**: Balanced, synthetic samples
